# Challenge: an adaptive time-stepping method

In [ ]:
#    APM41012EP course notebook - Chapter 7 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge: an adaptive time-stepping method
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Error estimator

We first show how to build an estimator of the local truncation error from the iterates of the explicit Euler method. The estimator makes it possible to predict an adaptive time step and to reduce the cost of the method at fixed accuracy. This estimator is tested on the Hirschfelder and Curtiss equation; it works well when the tolerance is small enough but raises problems when the tolerance is large. The goal of the challenge is then to explain this behaviour using the elements presented in the lecture.

Recall that, in the lecture, the local truncation error can be estimated from the second derivative of the solution $ \mathcal{E}_{n} = \frac{\Delta t_{n}^2}{2} y''(t_{n}) + O(\Delta t_{n}^3)$. When using an Euler-Cauchy or explicit Euler solver, show that an estimator of the local truncation error can be given by 

$$
 \mathcal{E}_{n} = \frac{\Delta t_{n}^2}{2 \Delta t_{n-1}}  \left ( f(t_{n}, y_{n}) - f(t_{n-1}, y_{n-1})\right ) + O(\Delta t_{n}^3), \qquad y''(t_{n}) \simeq \frac{f(t_{n}, y_{n}) - f(t_{n-1}, y_{n-1})}{\Delta t_{n-1}}.
$$

When the user wants to adapt the time step so as to obtain a local truncation error bounded by a tolerance $\epsilon$ of their choice, show that the time step at instant $n$ can be chosen by the formula: 

$$
\Delta t_{n} =   \sqrt{\frac{2\epsilon \Delta t_{n-1}}{|f(t_{n}, y_n) - f(t_{n-1}, y_{n-1})|} },
$$

provided that the user also supplies an initial time step. 

## Application to the Curtiss and Hirschfelder problem

We consider the following problem:

$$
\left\{ 
\begin{aligned} 
{\mathrm d}_t u(t) & = k \, \big(cos(t) - u(t)) \big) \quad \text{with } k > 1\\ 
u(0)  & = u_0 
\end{aligned} 
\right. 
$$

In [ ]:
class curtiss_model:

    def __init__(self, k):
        self.k = k

    def fcn(self, t, u) :
        k = self.k
        u_dot = k * (np.cos(t) - u)
        return u_dot

    def sol(self, uini, t0, t):
        k = self.k

        c0 = (uini - (k/(k*k + 1)) * (k*np.cos(t0) + np.sin(t0))) * np.exp(k*t0)
        u = (k/(k*k + 1)) * (k*np.cos(t) + np.sin(t)) +  c0 * np.exp(-k*t)
        return u

### Explicit Euler

In [ ]:
#####################################################
class ode_result:
    def __init__(self, y, t):
        self.y = y
        self.t = t

#####################################################
def forward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y[:,it+1] = yn + dt*np.array(fcn(tn, yn))

    return ode_result(y, t)

#####################################################
def forward_euler_adapted(tini, tend, dtini, yini, fcn, tol=1e-6):

    t = [tini];

    yini = np.atleast_1d(yini)
    y = [yini]
        
    dt = dtini
    tn = tini
    it = 0
    
    while (tn<tend):
        yn = y[it]
        y.append(yn + dt*np.array(fcn(tn, yn)))
        tn += dt
        t.append(tn)
        ddy = abs(y[it+1]-yn)/dt
        dt = np.sqrt(2*tol/np.linalg.norm(ddy))
        if (tn+dt)>tend: dt = tend-tn
        it += 1
        
    y = np.stack(y, axis=1)
    t = np.array(t)
    
    return ode_result(y, t)

#####################################################
def plot_fe_sol(k, tini, tend, nt, uini, tol):
    cm = curtiss_model(k)
    fcn = cm.fcn
    
    fig =  make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=("Solution", "Error"))
    
    sol = forward_euler(tini, tend, nt, uini, fcn)
    err = np.abs(sol.y[0] - cm.sol(uini, tini, sol.t))
    
    fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name=f'fixed dt, nt={sol.t.size}', 
                             line_color='rgb(76,114,176)', mode='markers+lines', legendgroup='1'), row=1, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=err, name=f'fixed dt, nt={sol.t.size}', 
                             line_color='rgb(76,114,176)', mode='markers+lines', legendgroup='2'), row=2, col=1)
    
    dtini = 1e-4
    nt_adapt = np.empty(len(tol), dtype=int)
    for i, tol_i in enumerate(tol):
        sol_adapt = forward_euler_adapted(tini, tend, dtini, uini, fcn, tol=tol_i)
        err_adapt = np.abs(sol_adapt.y[0] - cm.sol(uini, tini, sol_adapt.t))
        nt_adapt[i] = sol_adapt.t.size
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=sol_adapt.y[0], name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color='rgb(221,132,82)', mode='markers+lines', legendgroup='1'), row=1, col=1)
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=err_adapt, name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color='rgb(221,132,82)', mode='markers+lines', legendgroup='2'), row=2, col=1)
    
    fig.data[-2].visible = True
    fig.data[-1].visible = True
    
    # Create and add slider
    steps = []
    for i, tol_i in enumerate(tol):
        args = [{"visible": [(el==0) or (el==1) or (el==2*i+2) or (el==2*i+3) for el in range(len(fig.data))]}]
        step = dict(method="update", label = f"{tol_i:.2e}", args=args)
        steps.append(step)
    sliders = [dict(active=len(tol)-1, currentvalue={'prefix': 'Tolerance = '}, steps=steps)]

    for i, tol_i in enumerate(tol):
        print(f"Forward Euler with adaptive time step tol = {tol_i:.2e}:", end="")
        print(f"  nb of time points = {nt_adapt[i]}")

    legend=dict(x=0.70,y=0.98, bgcolor='rgba(0,0,0,0)')
    fig.update_layout(sliders=sliders, height=800, legend=legend, legend_tracegroupgap=300, legend_groupclick="toggleitem")
    fig.show()

In what follows, the reader can observe this method for several tolerances. One may notice that taking a tolerance of 10−5 leads to a good quality solution, with an error level lower than a solution with 400 points but with about twice fewer points. Choosing a larger tolerance of 10−4 leads to behaviours marred by large errors outside the initial time boundary layer. Explain this phenomenon.

In [ ]:
k = 100.

tini = 0.
tend = 0.5

nt = 400

uini = (2.,)

tol = [1.e-3, 1e-4, 1e-5, 1e-6]

plot_fe_sol(k, tini, tend, nt, uini, tol)